
# Buy Fee Model V12A — Chunked Buyer Behavior Analysis

## Purpose

This version is built specifically for the large buyer file:

`biweekly_buyer_churn_flags_detail_reactivated_loop1.csv`

It is designed to avoid the memory errors from earlier versions.

## Architecture

1. Read the buyer CSV in chunks.
2. Aggregate each chunk directly to:
   **Location × Biweekly period**
3. Combine those small aggregates.
4. Align buyer behavior to +$10 / +$15 / +$20 Buy Fee events.
5. Compare 4 biweekly periods before vs. 4 after each fee change.
6. Compare successful vs unsuccessful fee changes.
7. Rank which PRE-event buyer variables should be added to the next predictive model.

## Important

- The full buyer file is never loaded into memory.
- No weekly duplication is created.
- No sklearn / statsmodels / heavy ML is used.


In [1]:

from pathlib import Path
import pandas as pd
import numpy as np
import gc
import warnings

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================

DATA_FOLDER = Path(
    r"C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion"
)

V82_DIR = DATA_FOLDER / "buy_fee_model_v8_2_outputs"

DID_FILE = V82_DIR / "04_DID_EVENT_LEVEL_RESULTS.csv"

BUYER_FILE = DATA_FOLDER / "biweekly_buyer_churn_flags_detail_reactivated_loop1.csv"

OUTPUT_DIR = DATA_FOLDER / "buy_fee_model_v12a_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Buyer chunk size
CHUNK_SIZE = 100_000

SCENARIOS = [10, 15, 20]
SCENARIO_TOLERANCE = 2.5

# 4 biweekly periods ~= 8 weeks
PRE_BIWEEKLY_PERIODS = 4
POST_BIWEEKLY_PERIODS = 4

# Balanced success thresholds
RPU_SUCCESS_MIN = 0.0
CONVERSION_SAFE_MIN_PP = -2.0
VOLUME_SAFE_MIN_PCT = -0.10

print("=" * 80)
print("V12A INPUT CHECK")
print("=" * 80)

print("Buyer file:")
print(BUYER_FILE)
print("Exists:", BUYER_FILE.exists())

print("\nDiD file:")
print(DID_FILE)
print("Exists:", DID_FILE.exists())

if not BUYER_FILE.exists():
    raise FileNotFoundError(
        f"Buyer file not found:\n{BUYER_FILE}"
    )

if not DID_FILE.exists():
    raise FileNotFoundError(
        f"DiD file not found:\n{DID_FILE}"
    )

print("\nOutput folder:")
print(OUTPUT_DIR)


V12A INPUT CHECK
Buyer file:
C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\biweekly_buyer_churn_flags_detail_reactivated_loop1.csv
Exists: True

DiD file:
C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_model_v8_2_outputs\04_DID_EVENT_LEVEL_RESULTS.csv
Exists: True

Output folder:
C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_model_v12a_outputs


## 1. Inspect buyer columns

In [2]:

buyer_header = pd.read_csv(
    BUYER_FILE,
    nrows=5,
    low_memory=True,
    encoding="utf-8",
    encoding_errors="replace"
)

print("Buyer columns:")
for c in buyer_header.columns:
    print(" -", c)

def resolve_col(columns, aliases):
    lookup = {
        str(c).strip().upper(): c
        for c in columns
    }

    for alias in aliases:
        key = alias.strip().upper()
        if key in lookup:
            return lookup[key]

    return None

COL = {
    "location": resolve_col(
        buyer_header.columns,
        ["VRLOCATION", "LOCATION", "Location"]
    ),
    "buyer": resolve_col(
        buyer_header.columns,
        ["VRBUYER", "BUYER", "Buyer"]
    ),
    "start_date": resolve_col(
        buyer_header.columns,
        [
            "BiWeekly_Start_Date",
            "BIWEEKLY_START_DATE",
            "Biweekly_Start_Date"
        ]
    ),
    "end_date": resolve_col(
        buyer_header.columns,
        [
            "BiWeekly_End_Date",
            "BIWEEKLY_END_DATE",
            "Biweekly_End_Date"
        ]
    ),
    "active": resolve_col(
        buyer_header.columns,
        ["Active", "ACTIVE"]
    ),
    "fallout": resolve_col(
        buyer_header.columns,
        ["Fallout", "FALLOUT"]
    ),
    "dark": resolve_col(
        buyer_header.columns,
        ["Dark", "DARK"]
    ),
    "reactivated": resolve_col(
        buyer_header.columns,
        ["Reactivated", "REACTIVATED"]
    ),
    "net_new": resolve_col(
        buyer_header.columns,
        ["Net_New", "NET_NEW", "Net New"]
    ),
    "days_since_last": resolve_col(
        buyer_header.columns,
        [
            "Days_Since_Last_Purchase",
            "DAYS_SINCE_LAST_PURCHASE"
        ]
    ),
    "days_since_prev_loc": resolve_col(
        buyer_header.columns,
        [
            "Days_Since_Previous_Location_Purchase",
            "DAYS_SINCE_PREVIOUS_LOCATION_PURCHASE"
        ]
    ),
}

mapping = pd.DataFrame([
    {
        "Logical_Field": k,
        "Resolved_Column": v
    }
    for k, v in COL.items()
])

display(mapping)

required = [
    "location",
    "buyer",
    "start_date",
    "end_date"
]

missing = [
    k for k in required
    if COL[k] is None
]

if missing:
    raise ValueError(
        "Missing required buyer columns: "
        + ", ".join(missing)
    )


Buyer columns:
 - VRLOCATION
 - VRBUYER
 - BiWeekly_ID
 - BiWeekly_Start_Date
 - BiWeekly_End_Date
 - first_location_sale_date
 - Last_Purchase_Date
 - Previous_Location_Purchase_Date
 - Days_Since_Previous_Location_Purchase
 - Last_Purchase_Was_Reactivation
 - Days_Since_Last_Purchase
 - Lifecycle_Status
 - Active
 - Fallout
 - Dark
 - Reactivated
 - Net_New


,Logical_Field,Resolved_Column
0,location,VRLOCATION
1,buyer,VRBUYER
2,start_date,BiWeekly_Start_Date
3,end_date,BiWeekly_End_Date
4,active,Active
5,fallout,Fallout
6,dark,Dark
7,reactivated,Reactivated
8,net_new,Net_New
9,days_since_last,Days_Since_Last_Purchase


## 2. Chunked buyer aggregation

In [3]:

# ============================================================
# CHUNKED BUYER AGGREGATION
# ============================================================

USECOLS = [
    c for c in [
        COL["location"],
        COL["buyer"],
        COL["start_date"],
        COL["end_date"],
        COL["active"],
        COL["fallout"],
        COL["dark"],
        COL["reactivated"],
        COL["net_new"],
        COL["days_since_last"],
        COL["days_since_prev_loc"],
    ]
    if c is not None
]

def clean_text(s):
    return (
        s.astype("string")
        .fillna("")
        .str.strip()
    )

def to_flag(s):
    num = pd.to_numeric(
        s,
        errors="coerce"
    )

    out = pd.Series(
        np.zeros(len(s), dtype=np.int8),
        index=s.index
    )

    numeric_mask = num.notna()

    if numeric_mask.any():
        out.loc[numeric_mask] = (
            num.loc[numeric_mask]
            .gt(0)
            .astype("int8")
        )

    text_mask = ~numeric_mask

    if text_mask.any():
        txt = (
            s.loc[text_mask]
            .astype("string")
            .fillna("")
            .str.strip()
            .str.upper()
        )

        true_values = {
            "1", "Y", "YES", "TRUE",
            "ACTIVE", "FALLOUT", "DARK",
            "REACTIVATED", "NET NEW", "NET_NEW"
        }

        out.loc[text_mask] = (
            txt.isin(true_values)
            .astype("int8")
        )

    return out

partial_aggs = []

rows_read = 0

reader = pd.read_csv(
    BUYER_FILE,
    usecols=USECOLS,
    chunksize=CHUNK_SIZE,
    low_memory=True,
    encoding="utf-8",
    encoding_errors="replace"
)

for chunk_num, chunk in enumerate(reader, start=1):

    rows_read += len(chunk)

    loc = (
        clean_text(chunk[COL["location"]])
        .str.upper()
    )

    buyer_id = clean_text(
        chunk[COL["buyer"]]
    )

    start = pd.to_datetime(
        chunk[COL["start_date"]],
        errors="coerce"
    )

    end = pd.to_datetime(
        chunk[COL["end_date"]],
        errors="coerce"
    )

    valid = (
        loc.ne("")
        & buyer_id.ne("")
        & start.notna()
    )

    if not valid.any():
        del chunk
        gc.collect()
        continue

    temp = pd.DataFrame({
        "VRLOCATION": loc.loc[valid].values,
        "VRBUYER": buyer_id.loc[valid].values,
        "BiWeekly_Start": start.loc[valid].values,
        "BiWeekly_End": end.loc[valid].values,
    })

    for logical, outcol in [
        ("active", "Active_Flag"),
        ("fallout", "Fallout_Flag"),
        ("dark", "Dark_Flag"),
        ("reactivated", "Reactivated_Flag"),
        ("net_new", "Net_New_Flag"),
    ]:

        source = COL.get(logical)

        if source is not None:
            temp[outcol] = (
                to_flag(
                    chunk.loc[valid, source]
                )
                .values
            )
        else:
            temp[outcol] = np.int8(0)

    if COL["days_since_last"] is not None:
        temp["Days_Since_Last"] = pd.to_numeric(
            chunk.loc[
                valid,
                COL["days_since_last"]
            ],
            errors="coerce"
        ).astype("float32").values
    else:
        temp["Days_Since_Last"] = np.nan

    if COL["days_since_prev_loc"] is not None:
        temp["Days_Since_Prev_Loc"] = pd.to_numeric(
            chunk.loc[
                valid,
                COL["days_since_prev_loc"]
            ],
            errors="coerce"
        ).astype("float32").values
    else:
        temp["Days_Since_Prev_Loc"] = np.nan

    agg = (
        temp.groupby(
            [
                "VRLOCATION",
                "BiWeekly_Start",
                "BiWeekly_End"
            ],
            dropna=False,
            sort=False
        )
        .agg(
            Unique_Buyers=("VRBUYER", "nunique"),
            Active_Buyers=("Active_Flag", "sum"),
            Fallout_Buyers=("Fallout_Flag", "sum"),
            Dark_Buyers=("Dark_Flag", "sum"),
            Reactivated_Buyers=("Reactivated_Flag", "sum"),
            Net_New_Buyers=("Net_New_Flag", "sum"),

            Days_Since_Last_Sum=("Days_Since_Last", "sum"),
            Days_Since_Last_Count=("Days_Since_Last", "count"),

            Days_Since_Prev_Loc_Sum=("Days_Since_Prev_Loc", "sum"),
            Days_Since_Prev_Loc_Count=("Days_Since_Prev_Loc", "count"),
        )
        .reset_index()
    )

    partial_aggs.append(agg)

    if chunk_num % 10 == 0:
        print(
            f"Chunks={chunk_num:,} | "
            f"Rows read={rows_read:,} | "
            f"Partial aggregates={len(partial_aggs):,}"
        )

    del chunk, temp, agg
    gc.collect()

print("\nInitial chunk pass complete.")
print("Rows read:", f"{rows_read:,}")


Chunks=10 | Rows read=1,000,000 | Partial aggregates=10
Chunks=20 | Rows read=2,000,000 | Partial aggregates=20
Chunks=30 | Rows read=3,000,000 | Partial aggregates=30
Chunks=40 | Rows read=4,000,000 | Partial aggregates=40
Chunks=50 | Rows read=5,000,000 | Partial aggregates=50
Chunks=60 | Rows read=6,000,000 | Partial aggregates=60

Initial chunk pass complete.
Rows read: 6,679,333


## 3. Combine chunk-level buyer aggregates

In [4]:

buyer_period_raw = pd.concat(
    partial_aggs,
    ignore_index=True
)

del partial_aggs
gc.collect()

buyer_period = (
    buyer_period_raw.groupby(
        [
            "VRLOCATION",
            "BiWeekly_Start",
            "BiWeekly_End"
        ],
        dropna=False,
        sort=False
    )
    .agg(
        Unique_Buyers=("Unique_Buyers", "sum"),
        Active_Buyers=("Active_Buyers", "sum"),
        Fallout_Buyers=("Fallout_Buyers", "sum"),
        Dark_Buyers=("Dark_Buyers", "sum"),
        Reactivated_Buyers=("Reactivated_Buyers", "sum"),
        Net_New_Buyers=("Net_New_Buyers", "sum"),

        Days_Since_Last_Sum=("Days_Since_Last_Sum", "sum"),
        Days_Since_Last_Count=("Days_Since_Last_Count", "sum"),

        Days_Since_Prev_Loc_Sum=("Days_Since_Prev_Loc_Sum", "sum"),
        Days_Since_Prev_Loc_Count=("Days_Since_Prev_Loc_Count", "sum"),
    )
    .reset_index()
)

del buyer_period_raw
gc.collect()

buyer_period["Avg_Days_Since_Last_Purchase"] = np.where(
    buyer_period["Days_Since_Last_Count"] > 0,
    buyer_period["Days_Since_Last_Sum"]
    / buyer_period["Days_Since_Last_Count"],
    np.nan
)

buyer_period["Avg_Days_Since_Previous_Location_Purchase"] = np.where(
    buyer_period["Days_Since_Prev_Loc_Count"] > 0,
    buyer_period["Days_Since_Prev_Loc_Sum"]
    / buyer_period["Days_Since_Prev_Loc_Count"],
    np.nan
)

for num, rate in [
    ("Active_Buyers", "Active_Rate"),
    ("Fallout_Buyers", "Fallout_Rate"),
    ("Dark_Buyers", "Dark_Rate"),
    ("Reactivated_Buyers", "Reactivated_Rate"),
    ("Net_New_Buyers", "Net_New_Rate"),
]:
    buyer_period[rate] = np.where(
        buyer_period["Unique_Buyers"] > 0,
        buyer_period[num]
        / buyer_period["Unique_Buyers"],
        np.nan
    )

buyer_period.to_csv(
    OUTPUT_DIR / "01_BUYER_BIWEEKLY_AGGREGATE.csv",
    index=False
)

print("Buyer biweekly rows:", f"{len(buyer_period):,}")
print("Locations:", buyer_period["VRLOCATION"].nunique())
print(
    "Date range:",
    buyer_period["BiWeekly_Start"].min(),
    "to",
    buyer_period["BiWeekly_Start"].max()
)

display(buyer_period.head())


Buyer biweekly rows: 3,498
Locations: 49
Date range: 2019-12-30 00:00:00 to 2022-09-19 00:00:00


,VRLOCATION,BiWeekly_Start,BiWeekly_End,Unique_Buyers,Active_Buyers,Fallout_Buyers,Dark_Buyers,Reactivated_Buyers,Net_New_Buyers,Days_Since_Last_Sum,Days_Since_Last_Count,Days_Since_Prev_Loc_Sum,Days_Since_Prev_Loc_Count,Avg_Days_Since_Last_Purchase,Avg_Days_Since_Previous_Location_Purchase,Active_Rate,Fallout_Rate,Dark_Rate,Reactivated_Rate,Net_New_Rate
0,ALBANY,2019-12-30,2020-01-12,149,45,26,74,4,0,18634.0,149,4903.0,96,125.060403,51.072917,0.302013,0.174497,0.496644,0.026846,0.000000
1,ALBANY,2020-01-13,2020-01-26,149,50,22,72,5,0,18898.0,149,4707.0,98,126.832215,48.030612,0.335570,0.147651,0.483221,0.033557,0.000000
2,ALBANY,2020-01-27,2020-02-09,152,47,27,73,5,2,19997.0,152,4737.0,98,131.559211,48.336735,0.309211,0.177632,0.480263,0.032895,0.013158
3,ALBANY,2020-02-10,2020-02-23,154,46,29,76,3,1,21130.0,154,4469.0,98,137.207792,45.602041,0.298701,0.188312,0.493506,0.019481,0.006494
4,ALBANY,2020-02-24,2020-03-08,156,51,28,73,4,1,21703.0,156,4490.0,99,139.121795,45.353535,0.326923,0.179487,0.467949,0.025641,0.006410


## 4. Load matched Buy Fee events

In [5]:

events = pd.read_csv(
    DID_FILE,
    low_memory=True
)

events["Change_Date"] = pd.to_datetime(
    events["Corporate_Change_ID"],
    errors="coerce"
)

events["Treated_Location_STD"] = (
    events["Treated_Location"]
    .astype("string")
    .fillna("")
    .str.strip()
    .str.upper()
)

for c in [
    "BUYFEE_CHANGE_AMOUNT",
    "DiD_RPU_Change_Dollar",
    "DiD_Conversion_Change_pp",
    "DiD_Volume_Change_Pct"
]:
    events[c] = pd.to_numeric(
        events[c],
        errors="coerce"
    )

events["RPU_Success"] = (
    events["DiD_RPU_Change_Dollar"] > RPU_SUCCESS_MIN
)

events["Conversion_Safe"] = (
    events["DiD_Conversion_Change_pp"]
    >= CONVERSION_SAFE_MIN_PP
)

events["Volume_Safe"] = (
    events["DiD_Volume_Change_Pct"]
    >= VOLUME_SAFE_MIN_PCT
)

events["Balanced_Success"] = (
    events["RPU_Success"]
    & events["Conversion_Safe"]
    & events["Volume_Safe"]
)

scenario_mask = False

for s in SCENARIOS:
    scenario_mask = (
        scenario_mask
        |
        events["BUYFEE_CHANGE_AMOUNT"].between(
            s - SCENARIO_TOLERANCE,
            s + SCENARIO_TOLERANCE
        )
    )

events = events[
    scenario_mask
].copy()

print("Fee events retained:", f"{len(events):,}")
print("Locations:", events["Treated_Location_STD"].nunique())
print("Change dates:", events["Corporate_Change_ID"].nunique())


Fee events retained: 810
Locations: 24
Change dates: 29


## 5. Align buyer behavior to each fee event

In [6]:

metrics = [
    "Unique_Buyers",
    "Active_Buyers",
    "Fallout_Buyers",
    "Dark_Buyers",
    "Reactivated_Buyers",
    "Net_New_Buyers",
    "Active_Rate",
    "Fallout_Rate",
    "Dark_Rate",
    "Reactivated_Rate",
    "Net_New_Rate",
    "Avg_Days_Since_Last_Purchase",
    "Avg_Days_Since_Previous_Location_Purchase",
]

buyer_event_rows = []

for i, ev in events.iterrows():

    loc = ev["Treated_Location_STD"]
    change_date = ev["Change_Date"]

    loc_data = buyer_period[
        buyer_period["VRLOCATION"].eq(loc)
    ]

    if loc_data.empty or pd.isna(change_date):
        continue

    loc_data = loc_data.sort_values(
        "BiWeekly_Start"
    )

    midpoint = (
        loc_data["BiWeekly_Start"]
        +
        (
            loc_data["BiWeekly_End"]
            - loc_data["BiWeekly_Start"]
        ) / 2
    )

    pre = (
        loc_data[
            midpoint < change_date
        ]
        .tail(PRE_BIWEEKLY_PERIODS)
    )

    post = (
        loc_data[
            midpoint > change_date
        ]
        .head(POST_BIWEEKLY_PERIODS)
    )

    row = {
        "Segment_Change_ID": ev["Segment_Change_ID"],
        "Corporate_Change_ID": ev["Corporate_Change_ID"],
        "Change_Date": change_date,
        "Treated_Location": loc,
        "Treated_Lane": ev.get("Treated_Lane", np.nan),
        "Treated_Price_Tier": ev.get("Treated_Price_Tier", np.nan),
        "BUYFEE_CHANGE_AMOUNT": ev["BUYFEE_CHANGE_AMOUNT"],

        "DiD_RPU_Change_Dollar": ev["DiD_RPU_Change_Dollar"],
        "DiD_Conversion_Change_pp": ev["DiD_Conversion_Change_pp"],
        "DiD_Volume_Change_Pct": ev["DiD_Volume_Change_Pct"],

        "Balanced_Success": ev["Balanced_Success"],

        "Buyer_Pre_Periods": len(pre),
        "Buyer_Post_Periods": len(post),
    }

    for m in metrics:

        pre_val = (
            pre[m].mean()
            if len(pre)
            else np.nan
        )

        post_val = (
            post[m].mean()
            if len(post)
            else np.nan
        )

        row[f"Pre_{m}"] = pre_val
        row[f"Post_{m}"] = post_val

        row[f"Change_{m}"] = (
            post_val - pre_val
            if pd.notna(pre_val)
            and pd.notna(post_val)
            else np.nan
        )

        row[f"Pct_Change_{m}"] = (
            (post_val - pre_val) / pre_val
            if pd.notna(pre_val)
            and pre_val != 0
            and pd.notna(post_val)
            else np.nan
        )

    buyer_event_rows.append(row)

buyer_events = pd.DataFrame(
    buyer_event_rows
)

buyer_events.to_csv(
    OUTPUT_DIR / "02_BUYER_EVENT_LEVEL_ANALYSIS.csv",
    index=False
)

print("Buyer-linked events:", f"{len(buyer_events):,}")

print(
    "Full 4 PRE / 4 POST coverage:",
    (
        (buyer_events["Buyer_Pre_Periods"] >= PRE_BIWEEKLY_PERIODS)
        &
        (buyer_events["Buyer_Post_Periods"] >= POST_BIWEEKLY_PERIODS)
    ).sum()
)


Buyer-linked events: 783
Full 4 PRE / 4 POST coverage: 0


## 6. Buyer response by +$10 / +$15 / +$20

In [7]:

rows = []

for fee in SCENARIOS:

    d = buyer_events[
        buyer_events["BUYFEE_CHANGE_AMOUNT"].between(
            fee - SCENARIO_TOLERANCE,
            fee + SCENARIO_TOLERANCE
        )
    ]

    rows.append({
        "Fee_Increase": fee,
        "Events": len(d),
        "Locations": d["Treated_Location"].nunique(),
        "Corporate_Change_Dates": d["Corporate_Change_ID"].nunique(),

        "Median_Pct_Change_Unique_Buyers":
            d["Pct_Change_Unique_Buyers"].median(),

        "Median_Pct_Change_Active_Buyers":
            d["Pct_Change_Active_Buyers"].median(),

        "Median_Pct_Change_Fallout_Buyers":
            d["Pct_Change_Fallout_Buyers"].median(),

        "Median_Pct_Change_Dark_Buyers":
            d["Pct_Change_Dark_Buyers"].median(),

        "Median_Pct_Change_Reactivated_Buyers":
            d["Pct_Change_Reactivated_Buyers"].median(),

        "Median_Pct_Change_Net_New_Buyers":
            d["Pct_Change_Net_New_Buyers"].median(),

        "Median_Change_Active_Rate":
            d["Change_Active_Rate"].median(),

        "Median_Change_Fallout_Rate":
            d["Change_Fallout_Rate"].median(),

        "Median_Change_Dark_Rate":
            d["Change_Dark_Rate"].median(),
    })

scenario_buyer = pd.DataFrame(rows)

scenario_buyer.to_csv(
    OUTPUT_DIR / "03_BUYER_RESPONSE_BY_SCENARIO.csv",
    index=False
)

display(scenario_buyer)


,Fee_Increase,Events,Locations,Corporate_Change_Dates,Median_Pct_Change_Unique_Buyers,Median_Pct_Change_Active_Buyers,Median_Pct_Change_Fallout_Buyers,Median_Pct_Change_Dark_Buyers,Median_Pct_Change_Reactivated_Buyers,Median_Pct_Change_Net_New_Buyers,Median_Change_Active_Rate,Median_Change_Fallout_Rate,Median_Change_Dark_Rate
0,10,576,15,23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,15,84,14,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20,124,11,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 7. Successful vs unsuccessful PRE-event buyer profiles

In [8]:

profile_features = [
    "Pre_Unique_Buyers",
    "Pre_Active_Buyers",
    "Pre_Fallout_Buyers",
    "Pre_Dark_Buyers",
    "Pre_Reactivated_Buyers",
    "Pre_Net_New_Buyers",

    "Pre_Active_Rate",
    "Pre_Fallout_Rate",
    "Pre_Dark_Rate",
    "Pre_Reactivated_Rate",
    "Pre_Net_New_Rate",

    "Pre_Avg_Days_Since_Last_Purchase",
    "Pre_Avg_Days_Since_Previous_Location_Purchase",
]

profile_rows = []

for feature in profile_features:

    success = pd.to_numeric(
        buyer_events.loc[
            buyer_events["Balanced_Success"],
            feature
        ],
        errors="coerce"
    ).dropna()

    fail = pd.to_numeric(
        buyer_events.loc[
            ~buyer_events["Balanced_Success"],
            feature
        ],
        errors="coerce"
    ).dropna()

    profile_rows.append({
        "Feature": feature,
        "Successful_N": len(success),
        "Failed_N": len(fail),

        "Successful_Median":
            success.median()
            if len(success)
            else np.nan,

        "Failed_Median":
            fail.median()
            if len(fail)
            else np.nan,

        "Median_Difference":
            (
                success.median()
                - fail.median()
            )
            if len(success)
            and len(fail)
            else np.nan
    })

success_profile = pd.DataFrame(
    profile_rows
)

success_profile.to_csv(
    OUTPUT_DIR / "04_BUYER_SUCCESS_VS_FAILURE_PROFILE.csv",
    index=False
)

display(success_profile)


,Feature,Successful_N,Failed_N,Successful_Median,Failed_Median,Median_Difference
0,Pre_Unique_Buyers,236,547,2196.500000,2196.500000,0.000000
1,Pre_Active_Buyers,236,547,269.000000,283.500000,-14.500000
2,Pre_Fallout_Buyers,236,547,202.250000,202.250000,0.000000
3,Pre_Dark_Buyers,236,547,1584.500000,1584.500000,0.000000
4,Pre_Reactivated_Buyers,236,547,45.000000,45.000000,0.000000
5,Pre_Net_New_Buyers,236,547,1.250000,1.500000,-0.250000
6,Pre_Active_Rate,236,547,0.149258,0.149258,0.000000
7,Pre_Fallout_Rate,236,547,0.099457,0.092059,0.007398
8,Pre_Dark_Rate,236,547,0.736686,0.729100,0.007586
9,Pre_Reactivated_Rate,236,547,0.020496,0.020496,0.000000


## 8. Buyer mechanism correlations

In [9]:

mechanism_features = [
    "Pct_Change_Unique_Buyers",
    "Pct_Change_Active_Buyers",
    "Pct_Change_Fallout_Buyers",
    "Pct_Change_Dark_Buyers",
    "Pct_Change_Reactivated_Buyers",
    "Pct_Change_Net_New_Buyers",

    "Change_Active_Rate",
    "Change_Fallout_Rate",
    "Change_Dark_Rate",
]

mechanism_rows = []

for feature in mechanism_features:

    temp = buyer_events[
        [
            feature,
            "DiD_RPU_Change_Dollar",
            "DiD_Conversion_Change_pp",
            "DiD_Volume_Change_Pct"
        ]
    ].copy()

    for c in temp.columns:
        temp[c] = pd.to_numeric(
            temp[c],
            errors="coerce"
        )

    temp = temp.dropna(
        subset=[feature]
    )

    if len(temp) < 20:
        continue

    mechanism_rows.append({
        "Buyer_Feature": feature,
        "Events": len(temp),

        "Corr_with_RPU":
            temp[feature].corr(
                temp["DiD_RPU_Change_Dollar"]
            ),

        "Corr_with_Conversion":
            temp[feature].corr(
                temp["DiD_Conversion_Change_pp"]
            ),

        "Corr_with_Volume":
            temp[feature].corr(
                temp["DiD_Volume_Change_Pct"]
            ),
    })

mechanism = pd.DataFrame(
    mechanism_rows
)

mechanism.to_csv(
    OUTPUT_DIR / "05_BUYER_MECHANISM_CORRELATIONS.csv",
    index=False
)

display(mechanism)


""


## 9. Rank PRE-event buyer features for the next model

In [10]:

ranking_rows = []

for feature in profile_features:

    x = pd.to_numeric(
        buyer_events[feature],
        errors="coerce"
    )

    y_success = buyer_events[
        "Balanced_Success"
    ].astype(int)

    valid = x.notna()

    if valid.sum() < 20:
        continue

    def safe_corr(target_col):
        y = pd.to_numeric(
            buyer_events[target_col],
            errors="coerce"
        )

        mask = valid & y.notna()

        if mask.sum() < 20:
            return np.nan

        return x[mask].corr(
            y[mask]
        )

    ranking_rows.append({
        "Feature": feature,
        "Events": int(valid.sum()),

        "Correlation_with_Balanced_Success":
            x[valid].corr(
                y_success[valid]
            ),

        "Correlation_with_RPU":
            safe_corr(
                "DiD_RPU_Change_Dollar"
            ),

        "Correlation_with_Conversion":
            safe_corr(
                "DiD_Conversion_Change_pp"
            ),

        "Correlation_with_Volume":
            safe_corr(
                "DiD_Volume_Change_Pct"
            ),
    })

ranking = pd.DataFrame(
    ranking_rows
)

if not ranking.empty:

    ranking["Max_Absolute_Outcome_Correlation"] = (
        ranking[
            [
                "Correlation_with_RPU",
                "Correlation_with_Conversion",
                "Correlation_with_Volume"
            ]
        ]
        .abs()
        .max(axis=1)
    )

    ranking = ranking.sort_values(
        "Max_Absolute_Outcome_Correlation",
        ascending=False
    )

ranking.to_csv(
    OUTPUT_DIR / "06_BUYER_FEATURE_RANKING_FOR_V12B.csv",
    index=False
)

display(ranking)


,Feature,Events,Correlation_with_Balanced_Success,Correlation_with_RPU,Correlation_with_Conversion,Correlation_with_Volume,Max_Absolute_Outcome_Correlation
1,Pre_Active_Buyers,783,-0.085487,-0.175540,-0.068324,-0.007330,0.175540
4,Pre_Reactivated_Buyers,783,-0.059645,-0.169586,-0.011018,-0.027056,0.169586
2,Pre_Fallout_Buyers,783,-0.041192,-0.125855,0.039585,-0.006814,0.125855
6,Pre_Active_Rate,783,-0.032941,-0.085969,-0.109995,0.004089,0.109995
0,Pre_Unique_Buyers,783,-0.051898,-0.098016,0.018967,-0.002165,0.098016
5,Pre_Net_New_Buyers,783,-0.011309,-0.066415,-0.087621,-0.007079,0.087621
8,Pre_Dark_Rate,783,0.013211,0.083942,0.075508,-0.003638,0.083942
10,Pre_Net_New_Rate,783,0.011776,-0.048350,-0.072998,-0.010073,0.072998
3,Pre_Dark_Buyers,783,-0.043037,-0.072614,0.029561,-0.000121,0.072614
11,Pre_Avg_Days_Since_Last_Purchase,783,-0.021938,-0.053410,0.070433,0.032970,0.070433


## 10. Final executive summary

In [11]:

full_window = buyer_events[
    (buyer_events["Buyer_Pre_Periods"] >= PRE_BIWEEKLY_PERIODS)
    &
    (buyer_events["Buyer_Post_Periods"] >= POST_BIWEEKLY_PERIODS)
]

summary = pd.DataFrame([
    {
        "Metric": "Buyer_Biweekly_Aggregate_Rows",
        "Value": len(buyer_period)
    },
    {
        "Metric": "Buyer_Locations",
        "Value": buyer_period["VRLOCATION"].nunique()
    },
    {
        "Metric": "Buyer_Linked_Fee_Events",
        "Value": len(buyer_events)
    },
    {
        "Metric": "Full_4_Pre_4_Post_Events",
        "Value": len(full_window)
    },
    {
        "Metric": "Linked_Locations",
        "Value": buyer_events["Treated_Location"].nunique()
    },
    {
        "Metric": "Linked_Corporate_Change_Dates",
        "Value": buyer_events["Corporate_Change_ID"].nunique()
    },
])

summary.to_csv(
    OUTPUT_DIR / "07_V12A_EXECUTIVE_SUMMARY.csv",
    index=False
)

print("=" * 80)
print("V12A COMPLETE")
print("=" * 80)

print("\nOutput folder:")
print(OUTPUT_DIR)

display(summary)


V12A COMPLETE

Output folder:
C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_model_v12a_outputs


,Metric,Value
0,Buyer_Biweekly_Aggregate_Rows,3498
1,Buyer_Locations,49
2,Buyer_Linked_Fee_Events,783
3,Full_4_Pre_4_Post_Events,0
4,Linked_Locations,23
5,Linked_Corporate_Change_Dates,28



## Files to share back

Please send:

1. `07_V12A_EXECUTIVE_SUMMARY.csv`
2. `03_BUYER_RESPONSE_BY_SCENARIO.csv`
3. `04_BUYER_SUCCESS_VS_FAILURE_PROFILE.csv`
4. `05_BUYER_MECHANISM_CORRELATIONS.csv`
5. `06_BUYER_FEATURE_RANKING_FOR_V12B.csv`
6. `02_BUYER_EVENT_LEVEL_ANALYSIS.csv`

## Next step after V12A

If PRE-event buyer features explain meaningful variation, V12B should add only those PRE-event buyer variables to the +$10 / +$15 / +$20 models and compare held-out `Corporate_Change_ID` accuracy directly against V11.
